# A6 Leakage-Safe Split (CPU, LOCKED)

Membuat split `train`/`validation`/`test` yang *leakage-safe* dari silver labels
melalui `sipature_ml.split.run_split`. Ikuti `docs/leakage-safe-split-baseline-report.md`
dan `docs/reproducibility-runbook.md` sebelum eksekusi.

Input: `data/processed/canonical_reviews.parquet` (notebook `02`) dan
`data/annotations/silver-v1.0.0.jsonl` (notebook `03`).
Output: `data/splits/*.jsonl` + `split_manifest_silver_v1.json` (terkunci).

**PENTING (locked-test policy):** split hanya boleh dibuat SEKALI. Notebook ini
menolak membuat ulang bila manifest sudah ada di Drive, dan test split tidak
boleh dibaca sebelum model/threshold dibekukan (A8).


## Step 1 — Mount Google Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"
DRIVE_ANNOTATION_DIR = DRIVE_ROOT / "data" / "annotations"
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"

PROJECT_DIR = Path("/content/hackathon/ml")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
ANNOTATION_DIR = PROJECT_DIR / "data" / "annotations"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"

DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"

print("Drive root:", DRIVE_ROOT)
print("Sumber canonical reviews:", DRIVE_PROCESSED_DIR / "canonical_reviews.parquet")
print("Sumber silver labels   :", DRIVE_ANNOTATION_DIR / "silver-v1.0.0.jsonl")
print("Split dir (lokal)      :", SPLIT_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber canonical reviews: /content/drive/MyDrive/SIPATURE/data/processed/canonical_reviews.parquet
Sumber silver labels   : /content/drive/MyDrive/SIPATURE/data/annotations/silver-v1.0.0.jsonl
Split dir (lokal)      : /content/hackathon/ml/data/splits


## Step 3 — Clone repository dari GitHub


In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 4 — Verifikasi commit terbaru (git log)


In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
6e5a8dc (HEAD -> main, origin/main, origin/HEAD) docs: add step markers to notebooks 01-04
ed403db feat: add leakage-safe split notebook
23c4963 docs: record notebook 03 completion and mark sampling/silver annotation artifacts done


## Step 5 — Install dependencies


In [5]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 748.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

## Step 6 — Verifikasi versi package


In [3]:
import numpy
import pandas
import pyarrow
import sklearn

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("Scikit-learn:", sklearn.__version__)


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
Scikit-learn: 1.7.2


## Step 7 — Copy input dari Drive (canonical + silver)


In [4]:
# Salin input (canonical_reviews + silver labels) dari Drive ke lokal.
import shutil
from pathlib import Path

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

inputs = [
    (DRIVE_PROCESSED_DIR / "canonical_reviews.parquet", PROCESSED_DIR / "canonical_reviews.parquet"),
    (DRIVE_ANNOTATION_DIR / "silver-v1.0.0.jsonl", ANNOTATION_DIR / "silver-v1.0.0.jsonl"),
]

for source, destination in inputs:
    assert source.is_file(), (
        f"Input tidak ditemukan di Drive: {source}\n"
        "Jalankan notebook 02 dan 03 terlebih dahulu."
    )
    shutil.copy2(source, destination)
    print("Disalin:", source.name, "->", destination)


Disalin: canonical_reviews.parquet -> /content/hackathon/ml/data/processed/canonical_reviews.parquet
Disalin: silver-v1.0.0.jsonl -> /content/hackathon/ml/data/annotations/silver-v1.0.0.jsonl


## Step 8 — Import modul sipature_ml


In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 9 — Load split config & validasi silver


In [6]:
from sipature_ml.config import load_config
from sipature_ml.annotation import validate_silver_records

split_config = load_config("split")

print("Split version :", split_config["split_version"])
print("Seed          :", split_config["seed"])
print("Group column  :", split_config["group_column"])
print("Ratios        :", split_config["ratios"])
print("Annotation file :", split_config["annotation_file"])
print("Canonical file  :", split_config["canonical_reviews_file"])
print("Algorithm candidates:", split_config["algorithm"]["candidates"])

# Quality gate: validasi silver JSONL sebelum split.
validation = validate_silver_records(ANNOTATION_DIR / split_config["annotation_file"])
print("\nSilver validation -> records:", validation["records"],
      "| invalid:", validation["invalid_records"])
assert validation["invalid_records"] == 0, f"Silver invalid: {validation['errors']}"
print("Silver JSONL valid.")


Split version : silver-split-1.0.0
Seed          : 42
Group column  : destination_id
Ratios        : {'train': 0.7, 'validation': 0.15, 'test': 0.15}
Annotation file : silver-v1.0.0.jsonl
Canonical file  : canonical_reviews.parquet
Algorithm candidates: 500

Silver validation -> records: 1320 | invalid: 0
Silver JSONL valid.


## Step 10 — Guard split (buat / salin yang terkunci)


In [7]:
# Guard locked split: jangan buat ulang bila manifest sudah ada di Drive.
import shutil
from pathlib import Path

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

drive_manifest = DRIVE_SPLIT_DIR / "split_manifest_silver_v1.json"

if drive_manifest.is_file():
    print("Split SUDAH terkunci di Drive. Menyalin split yang ada, TANPA membuat ulang.")
    for source in sorted(DRIVE_SPLIT_DIR.glob("*")):
        if source.is_file():
            shutil.copy2(source, SPLIT_DIR / source.name)
            print("Disalin dari Drive:", source.name)
else:
    from sipature_ml.split import run_split
    split_manifest = run_split(PROCESSED_DIR, ANNOTATION_DIR, SPLIT_DIR, REPORT_DIR)
    print("Split berhasil dibuat dan terkunci.")
    for split in ("train", "validation", "test"):
        dist = split_manifest["distribution"][split]
        print(f"  {split}: {dist['records']} records, {dist['destinations']} destinations")


Split berhasil dibuat dan terkunci.
  train: 922 records, 187 destinations
  validation: 196 records, 40 destinations
  test: 202 records, 40 destinations


## Step 11 — Tampilkan manifest split (distribution & leakage)


In [8]:
# Tampilkan manifest split (distribution + leakage).
import json
from pathlib import Path

manifest_path = SPLIT_DIR / "split_manifest_silver_v1.json"
assert manifest_path.is_file(), "Split manifest tidak ditemukan"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

print("Split version:", manifest["split_version"])
print("Reference label:", manifest["reference_label_type"])
print("Test is locked:", manifest["test_is_locked"])
print("Component count:", manifest["component_count"])
print("Multi-destination components:", manifest["multi_destination_components"])
print("Cross-destination repeated-text groups:", manifest["cross_destination_repeated_text_groups"])

print("\nDISTRIBUTION:")
for split in ("train", "validation", "test"):
    dist = manifest["distribution"][split]
    print(f"  {split}: {dist['records']} records, {dist['destinations']} destinations, "
          f"{dist['empty_label_records']} empty-label")

print("\nLEAKAGE (semua harus 0):")
print(json.dumps(manifest["validation"]["leakage_counts"], indent=2))


Split version: silver-split-1.0.0
Reference label: ai_assisted_weak_supervision_silver
Test is locked: True
Component count: 219
Multi-destination components: 10
Cross-destination repeated-text groups: 23

DISTRIBUTION:
  train: 922 records, 187 destinations, 348 empty-label
  validation: 196 records, 40 destinations, 73 empty-label
  test: 202 records, 40 destinations, 76 empty-label

LEAKAGE (semua harus 0):
{
  "train_test_destinations": 0,
  "train_test_duplicates": 0,
  "train_test_repeated_texts": 0,
  "train_test_reviews": 0,
  "train_validation_destinations": 0,
  "train_validation_duplicates": 0,
  "train_validation_repeated_texts": 0,
  "train_validation_reviews": 0,
  "validation_test_destinations": 0,
  "validation_test_duplicates": 0,
  "validation_test_repeated_texts": 0,
  "validation_test_reviews": 0
}


## Step 12 — Copy output ke Drive


In [9]:
# Salin split + manifest + report ke Drive (artefak persisten & terkunci).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (SPLIT_DIR, DRIVE_SPLIT_DIR),
    (REPORT_DIR, DRIVE_REPORT_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file() and source.name not in {"README.md"}:
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


Disalin: split_manifest_silver_v1.json -> /content/drive/MyDrive/SIPATURE/data/splits
Disalin: test_silver_v1.jsonl -> /content/drive/MyDrive/SIPATURE/data/splits
Disalin: train_silver_v1.jsonl -> /content/drive/MyDrive/SIPATURE/data/splits
Disalin: validation_silver_v1.jsonl -> /content/drive/MyDrive/SIPATURE/data/splits
Disalin: split_summary.json -> /content/drive/MyDrive/SIPATURE/reports


## Step 13 — Run summary (hash & locked-test reminder)


In [10]:
# ============================================================
# RUN SUMMARY — hash, path, dan locked-test reminder.
# ============================================================
from sipature_ml.manifest import sha256_file

print("SPLIT VERSION:", manifest["split_version"])
print("SEED:", manifest["seed"])
print("RATIOS:", manifest["ratios"])

print("\nSOURCE HASHES:")
print("  silver_sha256            :", manifest["sources"]["silver_sha256"])
print("  canonical_reviews_sha256 :", manifest["sources"]["canonical_reviews_sha256"])
print("  split_config_sha256      :", manifest["sources"]["split_config_sha256"])
print("  taxonomy_sha256          :", manifest["sources"]["taxonomy_sha256"])

print("\nOUTPUT SPLIT DIR  :", SPLIT_DIR)
print("DRIVE SPLIT DIR    :", DRIVE_SPLIT_DIR)
print("OUTPUT REPORT DIR  :", REPORT_DIR)

print("\nOUTPUT HASHES:")
for split, info in manifest["outputs"].items():
    print(f"  {info['path']}: {info['sha256']}")

print("\nREMINDER: test split TERKUNCI. Jangan dibaca sampai model & threshold dibekukan (A8).")


SPLIT VERSION: silver-split-1.0.0
SEED: 42
RATIOS: {'test': 0.15, 'train': 0.7, 'validation': 0.15}

SOURCE HASHES:
  silver_sha256            : 8838930b046def5303c89efb4f018d9a5d8a77cc2b142fa25d4c445f4d9d2610
  canonical_reviews_sha256 : 1dbbebbf995fcc0013320b09d2570a93992be8bde2e7cc9f043f2d8be149b162
  split_config_sha256      : eb12e16bddc3cda4a94b85bab5ecce7956de3a2a7ac41dc15653ccf505af3bed
  taxonomy_sha256          : 9840978b6c62613cb580ba8c736558fda12b688ae38cd248e01428daacd727f5

OUTPUT SPLIT DIR  : /content/hackathon/ml/data/splits
DRIVE SPLIT DIR    : /content/drive/MyDrive/SIPATURE/data/splits
OUTPUT REPORT DIR  : /content/hackathon/ml/artifacts/reports

OUTPUT HASHES:
  test_silver_v1.jsonl: edf650024fc2f74c5f3eea1bc04c3b909c52884849067987196fd8b795bb43ff
  train_silver_v1.jsonl: 31c31803be592a3c91576f42abbc3fcf562d2f82c126b2856848e88639ab3fc4
  validation_silver_v1.jsonl: 7c2f5f911ea33c6854ad1adc21e41a58befb6b24b31cb5ef2b8e03b7b771477c

REMINDER: test split TERKUNCI. Janga